In [ ]:
!pip install -q epyt-flow

In [ ]:
import pandas as pd

try:
    from epyt_flow.data.benchmarks import battledim
    carregar = battledim.load_data
except (ImportError, AttributeError):
    from epyt_flow.data.benchmarks import load_battledim_data as carregar

dfs = {}
for flag in (False, True):
    df = carregar(return_test_scenario=flag, download_dir="./bd_data", verbose=False)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    ano = int(df["Timestamp"].dt.year.mode()[0])
    dfs[ano] = df.set_index("Timestamp")
    print(f"return_test_scenario={flag} -> ano {ano} | shape {df.shape}")

for ano, df in dfs.items():
    print(f"\n== {ano} ==")
    print("início/fim:", df.index.min(), df.index.max())
    print("passos:", df.index.to_series().diff().value_counts().head(3).to_dict())
    print("duplicados:", df.index.duplicated().sum(), "| NaN total:", int(df.isna().sum().sum()))
    for p in ("Pressure_", "Flow_", "Level_", "Demand_"):
        c = [x for x in df.columns if x.startswith(p)]
        print(f"{p}: {len(c)} -> {c[:5]}{' ...' if len(c) > 5 else ''}")

In [ ]:
df18 = dfs[2018]
print([c for c in df18.columns if not c.startswith("Demand_")])
print(df18.describe().T.loc[lambda t: ~t.index.str.startswith("Demand_"), ["min", "max", "mean"]])

In [ ]:
import epyt_flow.data.benchmarks.battledim_data as d
print(d.LEAKS_CONFIG_TRAIN[:600])
print("----")
print(d.LEAKS_CONFIG_TEST[:300])

In [ ]:
import numpy as np
from google.colab import files

COLS = [c for c in dfs[2018].columns if not c.startswith("Demand_")]   # 33 + 3 + 1
LIMITES = {"Pressure_": (0, 100), "Flow_": (0, 500), "Level_": (0, 10)}
MAX_GAP = 3   # janelas (15 min)

def interpolar_curto(s):
    """Interpola só buracos de até MAX_GAP janelas; buracos maiores ficam NaN."""
    na = s.isna()
    grupo = (na != na.shift()).cumsum()
    tamanho = na.groupby(grupo).transform("sum")
    curto = na & (tamanho <= MAX_GAP)
    return s.where(~curto, s.interpolate(limit_area="inside"))

def limpar(df):
    t = df[COLS].copy()
    t.index.name = "timestamp"
    fora = 0
    for pref, (lo, hi) in LIMITES.items():
        c = [x for x in COLS if x.startswith(pref)]
        mask = (t[c] < lo) | (t[c] > hi)
        fora += int(mask.sum().sum())
        t[c] = t[c].mask(mask)
    return t.apply(interpolar_curto), fora

for ano in (2018, 2019):
    t, fora = limpar(dfs[ano])
    assert len(t) == 105120 and t.index.year.unique().tolist() == [ano]
    t.to_csv(f"scada_{ano}.csv.gz", compression="gzip")
    print(f"{ano}: {t.shape} | fora da faixa: {fora} | NaN após limpeza: {int(t.isna().sum().sum())}")

# sensors.csv
TIPO = {"Pressure": ("pressao", "m"), "Flow": ("vazao", "m3/h"), "Level": ("nivel", "m")}
PAPEL = {"Flow_p227": "entrada", "Flow_p235": "entrada", "Flow_PUMP_1": "bomba", "Level_T1": "tanque"}
sensors = pd.DataFrame([
    {"sensor_id": c, "tipo": TIPO[c.split("_", 1)[0]][0], "unidade": TIPO[c.split("_", 1)[0]][1],
     "elemento_rede": c.split("_", 1)[1], "papel": PAPEL.get(c, "")}
    for c in COLS
])
sensors.to_csv("sensors.csv", index=False)
print(sensors["tipo"].value_counts().to_dict())

# rótulos
def ler_vazamentos(texto):
    reg = []
    for linha in texto.strip().splitlines():
        p = [x.strip() for x in linha.split(",")]
        if len(p) != 6:
            print("linha ignorada:", linha); continue
        reg.append({"pipe": p[0], "start": pd.to_datetime(p[1]), "end": pd.to_datetime(p[2]),
                    "leak_size": float(p[3]), "type": p[4], "peak": pd.to_datetime(p[5])})
    return pd.DataFrame(reg).sort_values("start").reset_index(drop=True)

l18 = ler_vazamentos(d.LEAKS_CONFIG_TRAIN)
l18["repaired_in_2018"] = l18["end"] < pd.Timestamp("2018-12-31 23:55")
l18.to_csv("leaks_2018.csv", index=False)

l19 = ler_vazamentos(d.LEAKS_CONFIG_TEST)
l19.to_csv("leaks_2019.csv", index=False)

print(f"2018: {len(l18)} vazamentos, {int(l18.repaired_in_2018.sum())} reparados no ano")
print(f"2019: {len(l19)} vazamentos (só para avaliação; não abrir)")

In [ ]:
for f in ["scada_2018.csv.gz", "scada_2019.csv.gz", "sensors.csv", "leaks_2018.csv", "leaks_2019.csv"]:
    files.download(f)

In [ ]:
!ls -la bd_data
files.download("bd_data/2018_SCADA.xlsx")
files.download("bd_data/2019_SCADA.xlsx")

In [ ]:
!ls -la bd_data/BattLeDIM

In [ ]:
files.download("bd_data/BattLeDIM/2018_SCADA.xlsx")
files.download("bd_data/BattLeDIM/2019_SCADA.xlsx")